In [118]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM


template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="llama3.2", temperature=0.8, top_k=10)




In [ ]:
chain = prompt | model

chain.invoke({"question": "What is LangChain?"})

In [ ]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="password",
    refresh_schema=False
)

In [119]:
from langchain_core.documents import Document

text = """
Marie Curie, 7 November 1867 – 4 July 1934, was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity.
She was the first woman to win a Nobel Prize, the first person to win a Nobel Prize twice, and the only person to win a Nobel Prize in two scientific fields.
Her husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them the first-ever married couple to win the Nobel Prize and launching the Curie family legacy of five Nobel Prizes.
She was, in 1906, the first woman to become a professor at the University of Paris.
Also, Robin Williams.
"""
documents = [Document(page_content=text)]

In [120]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

no_schema = LLMGraphTransformer(llm=model, strict_mode=True)

data = no_schema.convert_to_graph_documents(documents)

In [121]:
data

[GraphDocument(nodes=[], relationships=[], source=Document(metadata={}, page_content='\nMarie Curie, 7 November 1867 – 4 July 1934, was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity.\nShe was the first woman to win a Nobel Prize, the first person to win a Nobel Prize twice, and the only person to win a Nobel Prize in two scientific fields.\nHer husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them the first-ever married couple to win the Nobel Prize and launching the Curie family legacy of five Nobel Prizes.\nShe was, in 1906, the first woman to become a professor at the University of Paris.\nAlso, Robin Williams.\n'))]

In [122]:

system_prompt_1 = """
    "# Knowledge Graph Instructions for GPT-4\n"
    "## 1. Overview\n"
    "You are a top-tier algorithm designed for extracting information in structured "
    "formats to build a knowledge graph.\n"
    "Try to capture as much information from the text as possible without "
    "sacrificing accuracy. Do not add any information that is not explicitly "
    "mentioned in the text.\n"
    "- **Nodes** represent entities and concepts.\n"
    "- The aim is to achieve simplicity and clarity in the knowledge graph, making it\n"
    "accessible for a vast audience.\n"
    "## 2. Labeling Nodes\n"
    "- **Consistency**: Ensure you use available types for node labels.\n"
    "Ensure you use basic or elementary types for node labels.\n"
    "- For example, when you identify an entity representing a person, "
    "always label it as **'person'**. Avoid using more specific terms "
    "like 'mathematician' or 'scientist'."
    "- **Node IDs**: Never utilize integers as node IDs. Node IDs should be "
    "names or human-readable identifiers found in the text.\n"
    "- **Relationships** represent connections between entities or concepts.\n"
    "Ensure consistency and generality in relationship types when constructing "
    "knowledge graphs. Instead of using specific and momentary types "
    "such as 'BECAME_PROFESSOR', use more general and timeless relationship types "
    "like 'PROFESSOR'. Make sure to use general and timeless relationship types!\n"
    "## 3. Coreference Resolution\n"
    "- **Maintain Entity Consistency**: When extracting entities, it's vital to "
    "ensure consistency.\n"
    'If an entity, such as "John Doe", is mentioned multiple times in the text '
    'but is referred to by different names or pronouns (e.g., "Joe", "he"),'
    "always use the most complete identifier for that entity throughout the "
    'knowledge graph. In this example, use "John Doe" as the entity ID.\n'
    "Remember, the knowledge graph should be coherent and easily understandable, "
    "so maintaining consistency in entity references is crucial.\n"
    "## 4. Strict Compliance\n"
    "Adhere to the rules strictly. Non-compliance will result in termination."""




system_prompt_2 = """
        "You are a top-tier algorithm designed for extracting information in "
        "structured formats to build a knowledge graph. Your task is to identify "
        "the entities and relations requested with the user prompt from a given "
        "text. You must generate the output in a JSON format containing a list "
        'with JSON objects. Each object should have the keys: "head", '
        '"head_type", "relation", "tail", and "tail_type". The "head" '
        "key must contain the text of the extracted entity with one of the types "
        "from the provided list in the user prompt.",
        f'The "head_type" key must contain the type of the extracted head entity, '
        f'The "relation" key must contain the type of relation between the "head" '
        f'and the "tail"'
        f'The "tail" key must represent the text of an extracted entity which is '
        f'the tail of the relation, and the "tail_type" key must contain the type '
        "Your task is to extract relationships from text strictly adhering "
        "to the provided schema. The relationships can only appear "
        "between specific node types are presented in the schema format "
        "like: (Entity1Type, RELATIONSHIP_TYPE, Entity2Type) /n"
        "Attempt to extract as many entities and relations as you can. Maintain "
        "Entity Consistency: When extracting entities, it's vital to ensure "
        'consistency. If an entity, such as "John Doe", is mentioned multiple '
        "times in the text but is referred to by different names or pronouns "
        '(e.g., "Joe", "he"), always use the most complete identifier for '
        "that entity. The knowledge graph should be coherent and easily "
        "understandable, so maintaining consistency in entity references is "
        "crucial."""

In [123]:

examples = """{
        "text": (
            "Adam is a software engineer in Microsoft since 2009, "
            "and last year he got an award as the Best Talent"
        ),
        "head": "Adam",
        "head_type": "Person",
        "relation": "WORKS_FOR",
        "tail": "Microsoft",
        "tail_type": "Company",
    },
    {
        "text": (
            "Adam is a software engineer in Microsoft since 2009, "
            "and last year he got an award as the Best Talent"
        ),
        "head": "Adam",
        "head_type": "Person",
        "relation": "HAS_AWARD",
        "tail": "Best Talent",
        "tail_type": "Award",
    },
    {
        "text": (
            "Microsoft is a tech company that provide "
            "several products such as Microsoft Word"
        ),
        "head": "Microsoft Word",
        "head_type": "Product",
        "relation": "PRODUCED_BY",
        "tail": "Microsoft",
        "tail_type": "Company",
    },
    {
        "text": "Microsoft Word is a lightweight app that accessible offline",
        "head": "Microsoft Word",
        "head_type": "Product",
        "relation": "HAS_CHARACTERISTIC",
        "tail": "lightweight app",
        "tail_type": "Characteristic",
    },
    {
        "text": "Microsoft Word is a lightweight app that accessible offline",
        "head": "Microsoft Word",
        "head_type": "Product",
        "relation": "HAS_CHARACTERISTIC",
        "tail": "accessible offline",
        "tail_type": "Characteristic",
    },"""

def generate_prompt(text):
    return f"""Based on the following example, extract entities and "
                f"relations from the provided text.\n\n",
                f"Your task is to extract relationships from text. The relationships can only appear "
                f"between specific node types are presented in the schema format "
                f"like: (Entity1Type, RELATIONSHIP_TYPE, Entity2Type) /n"
                f"Below are a number of examples of text and their extracted "
                f"entities and relationships."
                f"{examples}\n\nExtract this: {text}"""
    


In [124]:
from typing import List
from pydantic import BaseModel, Field

class NerNel(BaseModel):
    head: str = Field(
        description=(
            "extracted head entity like Microsoft, Apple, John. "
            "Must use human-readable unique identifier."
        )
    )
    head_type: str = Field(
        description="type of the extracted head entity like Person, Company, etc"
    )
    relation: str = Field(description="relation between the head and the tail entities")
    tail: str = Field(
        description=(
            "extracted tail entity like Microsoft, Apple, John. "
            "Must use human-readable unique identifier."
        )
    )
    tail_type: str = Field(
        description="type of the extracted tail entity like Person, Company, etc"
    )

class Relationships(BaseModel):
    relationships: List[NerNel]
    


In [ ]:
#GraphDocument(nodes=nodes, relationships=relationships, source=document)
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship



**NER/Linking Hyperparameter Values (Example)**

*   **Temperature:** 0.2 - 0.5
*   **Top_p:** 0.1 - 0.3
*   **Top_k:** 20 - 50
*   **Max_tokens:** 128 - 256
*   **Repetition_penalty:** 1.1 - 1.2
*   **Frequency_penalty:** 0.0 - 0.1
*   **Presence_penalty:** 0.0 - 0.1
*   **Stop_sequences:** ["\n", "."] (adjust based on data)
*   **Num_beams:** 1 - 3
*   **Length_penalty:** 0.5 - 1.0
*   **Early_stopping:** True
*   **Do_sample:** True
*   **Logprobs:** None
*   **No_repeat_ngram_size:** 2-3
*   **Typical_p:** 0.8 - 0.95


## Description
*   **Temperature:** Controls randomness in output.
*   **Top_p:** Nucleus sampling threshold.


1. **Probability Ranking:** The language model predicts the probability of *every* possible next token (word or 
sub-word) given the current context.
2. **Cumulative Probability:**  It then sorts these tokens by their probabilities in descending order and calculates 
the *cumulative* probability – that is, the sum of the probabilities as you go down the list.
3. **Threshold (p):** You set a threshold value 'p' (usually between 0.7 and 0.95).  This is the "Top-p" value.
4. **Selection:** The model considers only the smallest possible set of tokens whose cumulative probability exceeds 
'p'. In other words, it keeps adding tokens to its selection until the sum of their probabilities reaches that 
threshold.
5. **Sampling:**  Finally, it randomly samples the next token from *only* the tokens within this selected set.



*   **Top_k:** Selects the top k most likely tokens.
*   **Max_tokens/Max_length:** Maximum output length.
*   **Repetition_penalty:** Discourages repeating tokens.
*   **Frequency_penalty:** Penalizes frequent tokens.
*   **Presence_penalty:** Penalizes tokens simply for existing.
*   **Stop_sequences:** Sequences that halt generation.
*   **Num_beams:** Number of beams for beam search.
*   **Length_penalty:** Favors or disfavors longer sequences.
*   **Early_stopping:** Stops beam search early.
*   **Do_sample:** Enables sampling-based generation.
*   **Logprobs:** Outputs probabilities of generated tokens.
*   **No_repeat_ngram_size:** Prevents n-gram repetitions.
*   **Typical_p:** Controls the distribution of typical tokens.

1. **Calculating "Typicality":** This is the core of the method. For each token, the algorithm calculates a 
“typicality” score. This score reflects how predictable the token is given the context.  A token is considered 
"typical" if its probability is relatively high but not *extremely* high – it’s more predictable than a random token, 
but it’s not so predictable that it's boring.  The calculation involves comparing the token's probability to the 
average probability of tokens in a large corpus of training data.

2. **Setting a Threshold (p):** Like with Top-p, you specify a threshold value 'p' (usually between 0.7 and 0.95).

3. **Selecting Typical Tokens:** The algorithm selects the set of tokens whose typicality scores are above a certain 
threshold. This is a more sophisticated selection than simply taking the top *k* or considering tokens up to a 
cumulative probability *p*.  It prioritizes tokens that are considered both reasonably probable *and* contribute to 
diversity.

4. **Sampling:** Finally, the model samples the next token from within this set of "typical" tokens.

**Key Differences from Top-p:**

* **Focus on Typicality:** Top-p is based on cumulative probability; Typical-p focuses on a metric of “typicality” that 
aims to capture how representative a token is of the training data.
* **More Complex Selection:** The token selection process in Typical-p is more intricate, as it involves calculating 
and comparing typicality scores.

**Why use Typical-p?**

* **Improved Quality:** Can often produce higher-quality text by avoiding both very predictable and very unpredictable 
tokens.
* **Enhanced Diversity:**  Aims for a better balance between coherence and creativity.



In [125]:
from ollama import Client
from pydantic import BaseModel
import json

client=Client(host='http://localhost:11434')

class LlmConfig:

    def __init__(self, **kwargs):
        # Set default values
        self.model = kwargs.get('model', 'llama3.2')
        self.temperature = kwargs.get('temperature', 0.0)
        self.top_k = kwargs.get('top_k', 1)
        self.top_p = kwargs.get('top_p', None)
        self.max_tokens = kwargs.get('max_tokens', 40)
        self.repeat_penalty = kwargs.get('repeat_penalty', None)
        self.frequency_penalty = kwargs.get('frequency_penalty', None)
        self.presence_penalty = kwargs.get('presence_penalty', None)
        self.typical_p = kwargs.get('typical_p', None)
        self.num_thread = kwargs.get('num_thread', None)
        

    # Attributes for type hinting and documentation
    model: str
    temperature: float
    top_k: int
    top_p: float
    max_tokens: int
    repeat_penalty: float
    frequency_penalty: float
    presence_penalty: float
    typical_p: float
    num_thread: int

In [126]:
llm_config = LlmConfig(model = "llama3.2", 
                       temperature = 0.5, 
                       top_p = 0.3, 
                       typical_p = 0.9,
                       top_k = 50, 
                       max_tokens = 256,
                       repeat_penalty = 1.2,
                       frequency_penalty = 0.1,
                       presence_penalty = 0.1,
                       num_thread = 16
                       )

def inference(prompt: str, system: str, config: LlmConfig):
    options = {
        "top_k": config.top_k,
        "top_p": config.top_p,
        "max_tokens": config.max_tokens,
        "temperature": config.temperature,
        "repeat_penalty": config.repeat_penalty,
        "frequency_penalty": config.frequency_penalty,
        "typical_p": config.typical_p,
        "num_thread": config.num_thread,
    }
    response = client.generate(
                            prompt=prompt,
                            system=system,
                            model=config.model,
                            format=Relationships.model_json_schema(),
                            options = options)
    return Relationships.model_validate_json(response.response)


In [127]:
result = inference(
    prompt=generate_prompt("John ran aways to get the ball and instead found a bag of beans. It was actually jack"), 
    system=system_prompt_2, 
    config=llm_config)

In [128]:
result

Relationships(relationships=[NerNel(head='John', head_type='Person', relation='RUNS_AWAY_FROM', tail='ball', tail_type='Object'), NerNel(head='John', head_type='Person', relation='FINDS', tail='bag of beans', tail_type='Entity')])

In [132]:
from py2neo import Graph, Node, Relationship

class CustomGraph:
    def __init__(self, **kwargs):
        # Set default values
        self.nodes = kwargs.get('nodes', [])
        self.relationships = kwargs.get('relationships', [])
        self.document = kwargs.get('document', "test")
        
    nodes: List[Node]
    relationships: List[Relationship]
    document: str
    

def create_graph_document(result: Relationships):
    nodes = []
    relationships = []

    for rel in result.relationships:
        # Construct nodes for head and tail
        head_node = Node(rel.head_type, title=rel.head)
        tail_node = Node(rel.tail_type, title=rel.tail)

        # Build Relationship correctly
        relationships.append(
            Relationship(
                head_node,
                rel.relation,
                tail_node
            )
        )

    return CustomGraph(nodes=nodes, relationships=relationships)

In [135]:
graph_document = create_graph_document(result)

In [136]:
graph = Graph("neo4j://localhost:7687", auth=("neo4j", "password"))
for node in graph_document.nodes:
    graph.create(node)
    
for relationship in graph_document.relationships:
    graph.create(relationship)

In [ ]:
data = Relationships(
    relationships=[
        NerNel(
            head="John", head_type="Person",
            relation="THROWS",
            tail="ball", tail_type="Object"
        ),
        NerNel(
            head="John", head_type="Person",
            relation="RUNS_AWAY_FROM",
            tail="bag of sand", tail_type="Location/Item"
        ),
    ]
)

graph_doc = create_graph_document(data)
print(graph_doc)

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_community.graphs.graph_document import Node, Relationship, GraphDocument

class NerNel(BaseModel):
    head: str = Field(description="extracted head entity")
    head_type: str = Field(description="type of the extracted head entity")
    relation: str = Field(description="relation between the head and the tail entities")
    tail: str = Field(description="extracted tail entity")
    tail_type: str = Field(description="type of the extracted tail entity")

class Relationships(BaseModel):
    relationships: List[NerNel]

# GraphDocument construction
def create_graph_document(result: Relationships):
    nodes_set = set()
    relationships = []
    
    graph = Graph("neo4j://localhost:7687", auth=("neo4j", "password"))
 
    person = Node("Person", name="John Doe")
    #graph.create(person)


    # Loop through the relationships in the result
    for rel in result.relationships:
        # Construct nodes for head and tail
        head_node = Node(rel.head_type, title=rel.head)
        tail_node = Node(rel.tail_type, title=rel.tail)

        # Build Relationship correctly
        relationships.append(
            Relationship(
                head_node,
                rel.relation,
                tail_node
            )
        )

    # Construct final list of Node objects
    nodes = [Node(id=node_id, type=node_type) for node_id, node_type in nodes_set]

    # Return the GraphDocument
    return GraphDocument(nodes=nodes)

# Test data
data = Relationships(
    relationships=[
        NerNel(
            head="John", head_type="Person",
            relation="THROWS",
            tail="ball", tail_type="Object"
        ),
        NerNel(
            head="John", head_type="Person",
            relation="RUNS_AWAY_FROM",
            tail="bag of sand", tail_type="Location/Item"
        ),
    ]
)

# Create the graph document
graph_doc = create_graph_document(data)
print(graph_doc)


In [ ]:
from typing import List
from pydantic import BaseModel, Field

from langchain_core.documents import Document

document = Document(
    page_content="Hello, world!",
    metadata={"source": "https://example.com"}
)

head_node = Node(id='dude', type='PERSON')
tail_node = Node(id='car', type='OBJECT')

relationship = Relationship(
                source=head_node,
                target=tail_node,
                type="WHERE_IS_MY_CAR"
            )

GraphDocument(nodes=[head_node, tail_node], relationships=[relationship], document=document)

In [ ]:
from py2neo import Graph, Node, Relationship

# Create a graph object
graph = Graph("neo4j://localhost:7687", auth=("neo4j", "password"))
 
person = Node("Person", name="John Doe")
graph.create(person)

movie = Node("Movie", title="The Matrix")
graph.create(movie)
person_movie = Relationship(person, "LOVES", movie)
graph.create(person_movie)

In [ ]:
from neo4j import GraphDatabase
 
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "password")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
     driver.verify_connectivity()